# 01 - Find some lightcurves

In [13]:
import lsdb
from lsdb import ConeSearch
from nested_pandas.utils import count_nested

lsdb.show_versions()


--------      SYSTEM INFO      --------
python        : 3.12.11
python-bits   : 64
OS            : Linux
OS-release    : 5.14.0-570.58.1.el9_6.x86_64
Version       : #1 SMP PREEMPT_DYNAMIC Fri Oct 31 13:55:05 UTC 2025
machine       : x86_64
processor     : 
byteorder     : little
LC_ALL        : 
LANG          : 
--------   INSTALLED VERSIONS   --------
lsdb          : 0.8.2
hats          : 0.8.2
nested-pandas : 0.6.8
pandas        : 2.3.1
numpy         : 2.4.2
dask          : 2025.7.0
pyarrow       : 17.0.0
fsspec        : 2025.7.0


In [14]:
object_columns = ['coord_dec', 'coord_ra', 
       'objectId', 
	'g_psfFlux',       'g_psfFluxErr', 
       'i_psfFlux', 'i_psfFluxErr',
       'r_psfFlux',       'r_psfFluxErr', 
       'u_psfFlux',       'u_psfFluxErr', 
       'y_psfFlux', 'y_psfFluxErr',
       'z_psfFlux', 'z_psfFluxErr',
       'objectForcedSource.band',
     'objectForcedSource.detector',
     'objectForcedSource.invalidPsfFlag',
     'objectForcedSource.midpointMjdTai',
     'objectForcedSource.pixelFlags_bad',
     'objectForcedSource.pixelFlags_cr',
     'objectForcedSource.pixelFlags_crCenter',
     'objectForcedSource.pixelFlags_edge',
     'objectForcedSource.pixelFlags_interpolated',
     'objectForcedSource.pixelFlags_interpolatedCenter',
     'objectForcedSource.pixelFlags_nodata',
     'objectForcedSource.pixelFlags_saturated',
     'objectForcedSource.pixelFlags_saturatedCenter',
     'objectForcedSource.pixelFlags_suspect',
     'objectForcedSource.pixelFlags_suspectCenter',
     'objectForcedSource.psfDiffFlux',
     'objectForcedSource.psfDiffFlux_flag',
     'objectForcedSource.psfDiffFluxErr',
     'objectForcedSource.psfFlux',
     'objectForcedSource.psfFlux_flag',
     'objectForcedSource.psfFluxErr']

In [15]:
objectForcedSource_flag_columns = ['objectForcedSource.invalidPsfFlag',
 'objectForcedSource.pixelFlags_bad',
 'objectForcedSource.pixelFlags_cr',
 'objectForcedSource.pixelFlags_crCenter',
 'objectForcedSource.pixelFlags_edge',
 'objectForcedSource.pixelFlags_interpolated',
 'objectForcedSource.pixelFlags_interpolatedCenter',
 'objectForcedSource.pixelFlags_nodata',
 'objectForcedSource.pixelFlags_saturated',
 'objectForcedSource.pixelFlags_saturatedCenter',
 'objectForcedSource.pixelFlags_suspect',
 'objectForcedSource.pixelFlags_suspectCenter',
 'objectForcedSource.psfDiffFlux_flag',
 'objectForcedSource.psfFlux_flag']

flag_query = " and ".join(objectForcedSource_flag_columns)
# flag_query

'objectForcedSource.invalidPsfFlag and objectForcedSource.pixelFlags_bad and objectForcedSource.pixelFlags_cr and objectForcedSource.pixelFlags_crCenter and objectForcedSource.pixelFlags_edge and objectForcedSource.pixelFlags_interpolated and objectForcedSource.pixelFlags_interpolatedCenter and objectForcedSource.pixelFlags_nodata and objectForcedSource.pixelFlags_saturated and objectForcedSource.pixelFlags_saturatedCenter and objectForcedSource.pixelFlags_suspect and objectForcedSource.pixelFlags_suspectCenter and objectForcedSource.psfDiffFlux_flag and objectForcedSource.psfFlux_flag'

In [16]:
count_query = " or ".join([f"n_lc_{band} >= 10" for band in 'ugrizy'])

In [17]:
cone = ConeSearch(52, -29, 250, fine=False)
ocat = lsdb.open_catalog('/sdf/data/rubin/shared/lsdb_commissioning/hats/v30_0_0_rc2/object_collection', columns=object_columns, search_filter=cone)
# ocat = lsdb.open_catalog('/sdf/data/rubin/shared/lsdb_commissioning/hats/v30_0_0_rc2/object_collection')
ocat

,coord_dec,coord_ra,objectId,g_psfFlux,g_psfFluxErr,i_psfFlux,i_psfFluxErr,r_psfFlux,r_psfFluxErr,u_psfFlux,u_psfFluxErr,y_psfFlux,y_psfFluxErr,z_psfFlux,z_psfFluxErr,objectForcedSource
npartitions=4,,,,,,,,,,,,,,,,
"Order: 9, Pixel: 2299635",double[pyarrow],double[pyarrow],int64[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],float[pyarrow],"nested<band: [string], detector: [int16], inva..."
"Order: 9, Pixel: 2299638",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Order: 9, Pixel: 2299641",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
"Order: 9, Pixel: 2299644",...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...


In [7]:
def find_lightcurves(some_data):
    # Count by band
    counts = count_nested(some_data, f"objectForcedSource", by="band", join=False)

    for band in 'ugrizy':
        
        if f"n_objectForcedSource_{band}" in counts.columns:
            # Drop light curves with less than 10 points
            some_data[f"lc_{band}"] = some_data[counts[f"n_objectForcedSource_{band}"] > 10].query(f"objectForcedSource.band=='{band}'")["objectForcedSource"]
            some_data = count_nested(some_data, f"lc_{band}", join=True)
    
        else:
            # there are no light curves - create an empty frame
            some_data[f"lc_{band}"] = some_data.query(f"objectForcedSource.band=='{band}'")["objectForcedSource"]
            some_data = count_nested(some_data, f"lc_{band}", join=True)

    # Drop objects that have NO light curves long enough
    some_data = some_data.query(count_query)
    
    # We don't need the flag colums anymore
    for col in objectForcedSource_flag_columns:
        some_data = some_data.drop(col, axis=1)

    # We also don't need the full list of sources - they're split to single-band light curves
    some_data = some_data.drop("objectForcedSource", axis=1)

    return some_data

In [8]:
ocat.map_partitions(find_lightcurves).write_catalog("./lcs", overwrite=True)

/tmp/ipykernel_9447/1382721711.py:1: DeprecationWarning: Call to deprecated method to_hats. (`to_hats` will be removed in the future, use `write_catalog` instead.) -- Deprecated since version 0.7.3.
  ocat.map_partitions(find_lightcurves).to_hats("./lcs", overwrite=True)
